<a href="https://colab.research.google.com/github/void-coder-x/DABA_SIG/blob/main/PS4_MITRE_Knowledge_Base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# PS4 — Build and Index the MITRE ATT&CK Knowledge Base

### SIG-DABA — Security Analytics

**Problem Statement:**
Build and index the MITRE ATT&CK knowledge base.

### Objective

This notebook collects MITRE ATT&CK Enterprise knowledge from the official
STIX dataset, extracts relevant security objects, converts them into
searchable documents, generates vector embeddings, and builds a FAISS
index for later retrieval.

### Workflow

MITRE ATT&CK STIX Data
→ Extract Techniques
→ Extract Mitigations
→ Extract Relationships
→ Create Knowledge Documents
→ Generate Embeddings
→ Build FAISS Index
→ Save Knowledge Base

In [ ]:
## 1. Scope

This stage focuses on constructing the MITRE ATT&CK knowledge base that
will be used by later RAG components.

### PS4 includes

- Downloading MITRE ATT&CK Enterprise STIX data
- Extracting attack techniques
- Extracting mitigations
- Extracting technique-mitigation relationships
- Creating structured knowledge documents
- Generating vector embeddings
- Building a FAISS similarity-search index
- Saving the index and metadata

### PS4 does not include

- Network anomaly detection
- Anomaly-to-technique mapping
- Security prompt design
- Complete RAG assistant
- LLM fine-tuning
- User interface

In [1]:
!pip -q install pandas numpy requests faiss-cpu transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 52.5 MB/s eta 0:00:00


In [2]:
import os
import json
import requests
import numpy as np
import pandas as pd
import faiss
import torch

from transformers import AutoTokenizer, AutoModel

print("Libraries imported successfully.")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Libraries imported successfully.
PyTorch version: 2.11.0+cpu
CUDA available: False


In [3]:
OUTPUT_DIR = "/content/PS4_results"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print("Output directory:", OUTPUT_DIR)

Output directory: /content/PS4_results


In [ ]:
## 2. MITRE ATT&CK Data Source

The knowledge base is constructed from the MITRE ATT&CK Enterprise STIX
dataset.

The STIX representation provides structured information about ATT&CK
techniques, mitigations, and relationships between ATT&CK objects.

In [4]:
MITRE_URL = (
    "https://raw.githubusercontent.com/mitre-attack/"
    "attack-stix-data/master/enterprise-attack/"
    "enterprise-attack.json"
)

response = requests.get(
    MITRE_URL,
    timeout=60
)

if response.status_code != 200:
    raise Exception(
        f"Failed to download MITRE ATT&CK data. "
        f"Status code: {response.status_code}"
    )

mitre_data = response.json()

print("MITRE ATT&CK data downloaded successfully.")
print("Total STIX objects:", len(mitre_data["objects"]))

MITRE ATT&CK data downloaded successfully.
Total STIX objects: 26086


In [5]:
from collections import Counter

object_types = Counter(
    obj.get("type")
    for obj in mitre_data["objects"]
)

object_types

Counter({'x-mitre-collection': 1,
         'campaign': 56,
         'course-of-action': 268,
         'intrusion-set': 191,
         'malware': 733,
         'tool': 95,
         'attack-pattern': 858,
         'x-mitre-matrix': 1,
         'x-mitre-tactic': 15,
         'x-mitre-analytic': 1758,
         'x-mitre-data-component': 109,
         'x-mitre-data-source': 38,
         'x-mitre-detection-strategy': 699,
         'relationship': 21262,
         'identity': 1,
         'marking-definition': 1})

In [6]:
techniques = []

for obj in mitre_data["objects"]:

    if obj.get("type") != "attack-pattern":
        continue

    if obj.get("revoked", False):
        continue

    if obj.get("x_mitre_deprecated", False):
        continue

    technique_id = None

    for ref in obj.get("external_references", []):

        if ref.get("source_name") == "mitre-attack":

            external_id = ref.get(
                "external_id",
                ""
            )

            if external_id.startswith("T"):
                technique_id = external_id
                break

    if technique_id:

        techniques.append({
            "STIX_ID": obj.get("id"),
            "Technique_ID": technique_id,
            "Technique_Name": obj.get("name", ""),
            "Description": obj.get("description", ""),
            "Created": obj.get("created"),
            "Modified": obj.get("modified")
        })

techniques_df = pd.DataFrame(techniques)

print(
    "Techniques extracted:",
    len(techniques_df)
)

display(techniques_df.head())

Techniques extracted: 697


,STIX_ID,Technique_ID,Technique_Name,Description,Created,Modified
0,attack-pattern--0042a9f5-f053-4769-b3ef-9ad018...,T1055.011,Extra Window Memory Injection,Adversaries may inject malicious code into pro...,2020-01-14T17:18:32.126Z,2026-05-12T15:12:00.617Z
1,attack-pattern--005a06c6-14bf-4118-afa0-ebcd8a...,T1053.005,Scheduled Task,Adversaries may abuse the Windows Task Schedul...,2019-11-27T14:58:00.429Z,2026-05-12T15:12:00.618Z
2,attack-pattern--005cc321-08ce-4d17-b1ea-cb5275...,T1205.002,Socket Filters,Adversaries may attach filters to a network so...,2022-09-30T21:18:41.930Z,2026-05-12T15:12:00.619Z
3,attack-pattern--00f90846-cbd1-4fc5-9233-df5c2b...,T1560.001,Archive via Utility,Adversaries may use utilities to compress and/...,2020-02-20T21:01:25.428Z,2026-05-12T15:12:00.619Z
4,attack-pattern--01327cde-66c4-4123-bf34-5f258d...,T1021.005,VNC,Adversaries may use [Valid Accounts](https://a...,2020-02-11T18:28:44.950Z,2025-10-24T17:48:19.567Z


In [7]:
mitigations = []

for obj in mitre_data["objects"]:

    if obj.get("type") != "course-of-action":
        continue

    if obj.get("revoked", False):
        continue

    if obj.get("x_mitre_deprecated", False):
        continue

    mitigation_id = None

    for ref in obj.get("external_references", []):

        if ref.get("source_name") == "mitre-attack":

            external_id = ref.get(
                "external_id",
                ""
            )

            if external_id.startswith("M"):
                mitigation_id = external_id
                break

    if mitigation_id:

        mitigations.append({
            "STIX_ID": obj.get("id"),
            "Mitigation_ID": mitigation_id,
            "Mitigation_Name": obj.get("name", ""),
            "Description": obj.get("description", ""),
            "Created": obj.get("created"),
            "Modified": obj.get("modified")
        })

mitigations_df = pd.DataFrame(mitigations)

print(
    "Mitigations extracted:",
    len(mitigations_df)
)

display(mitigations_df.head())

Mitigations extracted: 44


,STIX_ID,Mitigation_ID,Mitigation_Name,Description,Created,Modified
0,course-of-action--12241367-a8b7-49b4-b86e-2236...,M1031,Network Intrusion Prevention,Use intrusion detection signatures to block tr...,2019-06-10T20:46:02.263Z,2024-10-17T18:54:36.723Z
1,course-of-action--15437c6d-b998-4a36-be41-4ace...,M1016,Vulnerability Scanning,Vulnerability scanning involves the automated ...,2019-06-06T16:47:30.700Z,2024-12-24T14:41:01.585Z
2,course-of-action--1dcaeb21-9348-42ea-950a-f842...,M1035,Limit Access to Resource Over Network,"Restrict access to network resources, such as ...",2019-06-11T16:30:16.672Z,2024-12-18T15:50:51.212Z
3,course-of-action--20a2baeb-98c2-4901-bad7-dc62...,M1029,Remote Data Storage,Remote Data Storage focuses on moving critical...,2019-06-06T21:21:13.027Z,2024-12-18T19:03:10.800Z
4,course-of-action--20f6a9df-37c4-4e20-9e47-0259...,M1037,Filter Network Traffic,Employ network appliances and endpoint softwar...,2019-06-11T16:33:55.337Z,2026-05-12T15:12:00.730Z


In [8]:
relationships = []

for obj in mitre_data["objects"]:

    if obj.get("type") != "relationship":
        continue

    if obj.get("relationship_type") != "mitigates":
        continue

    source_ref = obj.get("source_ref", "")
    target_ref = obj.get("target_ref", "")

    relationships.append({
        "STIX_ID": obj.get("id"),
        "Source": source_ref,
        "Target": target_ref,
        "Relationship": "mitigates",
        "Created": obj.get("created"),
        "Modified": obj.get("modified")
    })

relationships_df = pd.DataFrame(
    relationships
)

print(
    "Mitigation relationships:",
    len(relationships_df)
)

display(relationships_df.head())

Mitigation relationships: 1448


,STIX_ID,Source,Target,Relationship,Created,Modified
0,relationship--0c2dadcb-36c4-4255-a53d-e54f1bb7...,course-of-action--12241367-a8b7-49b4-b86e-2236...,attack-pattern--7bd9c723-2f78-4309-82c5-47cad4...,mitigates,2020-03-12T14:18:38.906Z,2026-04-29T14:23:05.920Z
1,relationship--123c8150-c814-4bf2-b06f-e005cb25...,course-of-action--12241367-a8b7-49b4-b86e-2236...,attack-pattern--28abec6c-4443-4b03-8206-07f2e2...,mitigates,2020-10-20T17:59:20.709Z,2026-04-29T14:23:05.929Z
2,relationship--1469ff36-c655-4513-8ffa-81dcb55c...,course-of-action--12241367-a8b7-49b4-b86e-2236...,attack-pattern--f24faf46-3b26-4dbb-98f2-634604...,mitigates,2019-06-21T14:29:50.951Z,2026-04-29T14:23:05.932Z
3,relationship--17639fa3-3f77-4481-b24f-9df82fcb...,course-of-action--12241367-a8b7-49b4-b86e-2236...,attack-pattern--4fe28b27-b13c-453e-a386-c2ef36...,mitigates,2020-03-15T16:03:39.234Z,2026-04-29T14:23:05.936Z
4,relationship--19f35c88-92c4-4e98-b024-be9f6731...,course-of-action--12241367-a8b7-49b4-b86e-2236...,attack-pattern--355be19c-ffc9-46d5-8d50-d6a036...,mitigates,2019-06-21T15:16:29.301Z,2026-04-29T14:23:05.940Z


In [9]:
technique_lookup = dict(
    zip(
        techniques_df["STIX_ID"],
        techniques_df["Technique_ID"] + " - " +
        techniques_df["Technique_Name"]
    )
)

mitigation_lookup = dict(
    zip(
        mitigations_df["STIX_ID"],
        mitigations_df["Mitigation_ID"] + " - " +
        mitigations_df["Mitigation_Name"]
    )
)

print(
    "Technique lookup entries:",
    len(technique_lookup)
)

print(
    "Mitigation lookup entries:",
    len(mitigation_lookup)
)

Technique lookup entries: 697
Mitigation lookup entries: 44


In [10]:
resolved_relationships = []

for _, row in relationships_df.iterrows():

    mitigation = mitigation_lookup.get(
        row["Source"]
    )

    technique = technique_lookup.get(
        row["Target"]
    )

    if mitigation and technique:

        resolved_relationships.append({
            "Mitigation": mitigation,
            "Technique": technique,
            "Relationship": "mitigates"
        })

resolved_relationships_df = pd.DataFrame(
    resolved_relationships
)

print(
    "Resolved relationships:",
    len(resolved_relationships_df)
)

display(
    resolved_relationships_df.head(10)
)

Resolved relationships: 1448


,Mitigation,Technique,Relationship
0,M1031 - Network Intrusion Prevention,T1568 - Dynamic Resolution,mitigates
1,M1031 - Network Intrusion Prevention,T1542.005 - TFTP Boot,mitigates
2,M1031 - Network Intrusion Prevention,T1008 - Fallback Channels,mitigates
3,M1031 - Network Intrusion Prevention,T1572 - Protocol Tunneling,mitigates
4,M1031 - Network Intrusion Prevention,T1071 - Application Layer Protocol,mitigates
5,M1031 - Network Intrusion Prevention,T1048.003 - Exfiltration Over Unencrypted Non-...,mitigates
6,M1031 - Network Intrusion Prevention,T1105 - Ingress Tool Transfer,mitigates
7,M1031 - Network Intrusion Prevention,T1132 - Data Encoding,mitigates
8,M1031 - Network Intrusion Prevention,T1568.002 - Domain Generation Algorithms,mitigates
9,M1031 - Network Intrusion Prevention,T1102 - Web Service,mitigates


In [11]:
documents = []

for _, row in techniques_df.iterrows():

    text = (
        f"MITRE ATT&CK Technique: "
        f"{row['Technique_ID']} - {row['Technique_Name']}\n\n"
        f"Description:\n"
        f"{row['Description']}"
    )

    documents.append({
        "doc_id": row["STIX_ID"],
        "type": "technique",
        "external_id": row["Technique_ID"],
        "name": row["Technique_Name"],
        "text": text
    })

print(
    "Technique documents:",
    len(documents)
)

Technique documents: 697


In [12]:
for _, row in mitigations_df.iterrows():

    text = (
        f"MITRE ATT&CK Mitigation: "
        f"{row['Mitigation_ID']} - "
        f"{row['Mitigation_Name']}\n\n"
        f"Description:\n"
        f"{row['Description']}"
    )

    documents.append({
        "doc_id": row["STIX_ID"],
        "type": "mitigation",
        "external_id": row["Mitigation_ID"],
        "name": row["Mitigation_Name"],
        "text": text
    })

print(
    "Total knowledge documents:",
    len(documents)
)

Total knowledge documents: 741


In [13]:
for _, row in resolved_relationships_df.iterrows():

    text = (
        f"MITRE ATT&CK relationship\n"
        f"Technique: {row['Technique']}\n"
        f"Mitigation: {row['Mitigation']}\n"
        f"Relationship: {row['Relationship']}"
    )

    documents.append({
        "doc_id": (
            f"{row['Technique']}_"
            f"{row['Mitigation']}"
        ),
        "type": "relationship",
        "external_id": "",
        "name": "Technique-Mitigation Relationship",
        "text": text
    })

print(
    "Final knowledge document count:",
    len(documents)
)

Final knowledge document count: 2189


In [14]:
for doc in documents[:3]:

    print("=" * 70)
    print(doc["type"])
    print(doc["name"])
    print(doc["text"][:1000])

technique
Extra Window Memory Injection
MITRE ATT&CK Technique: T1055.011 - Extra Window Memory Injection

Description:
Adversaries may inject malicious code into process via Extra Window Memory (EWM) in order to evade process-based defenses as well as possibly elevate privileges. EWM injection is a method of executing arbitrary code in the address space of a separate live process. 

Before creating a window, graphical Windows-based processes must prescribe to or register a windows class, which stipulate appearance and behavior (via windows procedures, which are functions that handle input/output of data).(Citation: Microsoft Window Classes) Registration of new windows classes can include a request for up to 40 bytes of EWM to be appended to the allocated memory of each instance of that class. This EWM is intended to store data specific to that window and has specific application programming interface (API) functions to set and get its value. (Citation: Microsoft GetWindowLong function

In [ ]:
## 3. Generate Vector Embeddings

The textual MITRE ATT&CK knowledge documents are converted into numerical
vectors so that semantically similar documents can be retrieved.

A pretrained Hugging Face transformer model is used to generate the
embeddings.

The model is used for embedding generation; it is **not fine-tuned on
MITRE ATT&CK data** in this notebook.

In [15]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModel.from_pretrained(
    MODEL_NAME
)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)
model.eval()

print("Embedding model loaded.")
print("Device:", device)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.
Device: cpu


In [16]:
def mean_pooling(
    model_output,
    attention_mask
):

    token_embeddings = model_output[0]

    input_mask_expanded = (
        attention_mask
        .unsqueeze(-1)
        .expand(token_embeddings.size())
        .float()
    )

    return torch.sum(
        token_embeddings * input_mask_expanded,
        dim=1
    ) / torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )

In [17]:
def generate_embeddings(
    texts,
    batch_size=32
):

    all_embeddings = []

    for start in range(
        0,
        len(texts),
        batch_size
    ):

        batch = texts[
            start:start + batch_size
        ]

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )

        encoded = {
            key: value.to(device)
            for key, value in encoded.items()
        }

        with torch.no_grad():

            model_output = model(
                **encoded
            )

        embeddings = mean_pooling(
            model_output,
            encoded["attention_mask"]
        )

        embeddings = torch.nn.functional.normalize(
            embeddings,
            p=2,
            dim=1
        )

        all_embeddings.append(
            embeddings.cpu().numpy()
        )

    return np.vstack(
        all_embeddings
    )

In [18]:
document_texts = [
    doc["text"]
    for doc in documents
]

embeddings = generate_embeddings(
    document_texts,
    batch_size=32
)

print(
    "Embedding shape:",
    embeddings.shape
)

Embedding shape: (2189, 384)


In [ ]:
## 4. Build FAISS Vector Index

FAISS is used to efficiently search the vector representations of MITRE
ATT&CK knowledge documents.

Because the embeddings are normalized, inner-product similarity can be
used as cosine similarity.

In [19]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(
    dimension
)

index.add(
    embeddings.astype(
        "float32"
    )
)

print(
    "FAISS index created."
)

print(
    "Indexed documents:",
    index.ntotal
)

print(
    "Vector dimension:",
    dimension
)

FAISS index created.
Indexed documents: 2189
Vector dimension: 384


In [20]:
test_query = (
    "network activity involving "
    "many connection attempts"
)

query_embedding = generate_embeddings(
    [test_query]
)

scores, indices = index.search(
    query_embedding.astype("float32"),
    5
)

print("Query:", test_query)
print()

for rank, (score, idx) in enumerate(
    zip(scores[0], indices[0]),
    start=1
):

    doc = documents[idx]

    print(
        f"{rank}. "
        f"{doc['type']} | "
        f"{doc['external_id']} | "
        f"{doc['name']}"
    )

    print(
        f"Similarity: {score:.4f}"
    )

    print()

Query: network activity involving many connection attempts

1. technique | T1498 | Network Denial of Service
Similarity: 0.4829

2. technique | T1016.001 | Internet Connection Discovery
Similarity: 0.4532

3. technique | T1110.003 | Password Spraying
Similarity: 0.4436

4. technique | T1498.001 | Direct Network Flood
Similarity: 0.4350

5. technique | T1049 | System Network Connections Discovery
Similarity: 0.4241



In [21]:
metadata = []

for doc in documents:

    metadata.append({
        "doc_id": doc["doc_id"],
        "type": doc["type"],
        "external_id": doc["external_id"],
        "name": doc["name"],
        "text": doc["text"]
    })

print(
    "Metadata records:",
    len(metadata)
)

Metadata records: 2189


In [22]:
INDEX_PATH = os.path.join(
    OUTPUT_DIR,
    "mitre_faiss.index"
)

faiss.write_index(
    index,
    INDEX_PATH
)

print(
    "FAISS index saved to:",
    INDEX_PATH
)

FAISS index saved to: /content/PS4_results/mitre_faiss.index


In [23]:
METADATA_PATH = os.path.join(
    OUTPUT_DIR,
    "mitre_documents.json"
)

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "Metadata saved to:",
    METADATA_PATH
)

Metadata saved to: /content/PS4_results/mitre_documents.json


In [24]:
techniques_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "mitre_techniques.csv"
    ),
    index=False
)

mitigations_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "mitre_mitigations.csv"
    ),
    index=False
)

resolved_relationships_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "mitre_relationships.csv"
    ),
    index=False
)

print("CSV knowledge-base files saved.")

CSV knowledge-base files saved.


In [25]:
print(
    "PS4 output files:"
)

for filename in os.listdir(
    OUTPUT_DIR
):

    filepath = os.path.join(
        OUTPUT_DIR,
        filename
    )

    size_mb = os.path.getsize(
        filepath
    ) / (1024 * 1024)

    print(
        f"- {filename}: "
        f"{size_mb:.2f} MB"
    )

PS4 output files:
- mitre_mitigations.csv: 0.09 MB
- mitre_relationships.csv: 0.10 MB
- mitre_documents.json: 1.69 MB
- mitre_faiss.index: 3.21 MB
- mitre_techniques.csv: 1.03 MB


In [26]:
summary = {
    "techniques": len(techniques_df),
    "mitigations": len(mitigations_df),
    "relationships": len(resolved_relationships_df),
    "documents": len(documents),
    "embedding_dimension": int(dimension),
    "faiss_vectors": int(index.ntotal)
}

summary

{'techniques': 697,
 'mitigations': 44,
 'relationships': 1448,
 'documents': 2189,
 'embedding_dimension': 384,
 'faiss_vectors': 2189}

In [27]:
SUMMARY_PATH = os.path.join(
    OUTPUT_DIR,
    "ps4_summary.json"
)

with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

print(
    "PS4 summary saved."
)

PS4 summary saved.


In [ ]:
## 5. Limitations

1. The knowledge base is based on the MITRE ATT&CK Enterprise STIX dataset
   available at the time of execution.

2. The embedding model is pretrained and is not specifically fine-tuned
   for cybersecurity.

3. Similarity search retrieves potentially relevant documents but does not
   establish that a technique is actually present in a network incident.

4. The FAISS index provides vector similarity retrieval; reasoning and
   response generation are handled in later RAG stages.

5. The knowledge base should be periodically rebuilt when the underlying
   MITRE ATT&CK dataset is updated.

In [ ]:
# Conclusion

The MITRE ATT&CK Enterprise knowledge base was successfully constructed
from structured STIX data.

The notebook extracted:

- ATT&CK techniques
- ATT&CK mitigations
- Technique-mitigation relationships

The extracted knowledge was converted into searchable documents,
embedded using a pretrained transformer model, and indexed using FAISS.

The resulting knowledge base provides the retrieval layer required by
later security RAG stages.